In [1]:
import os;

In [2]:
%pwd

'c:\\Users\\chait\\Parkinsons\\research'

In [3]:
os.chdir("../");  

In [4]:
from dataclasses import dataclass;
from pathlib import Path;

In [16]:
@dataclass
class ModelParams:
    input_nodes: int;
    hidden_layers: int;
    output_nodes: int;
    optimizer: str;
    loss: str;

@dataclass
class ModelConfig:
    data_source: Path;
    model_store: Path;

In [6]:
import pandas as pd;
import tensorflow as tf;
from tensorflow.keras import layers, models;


In [7]:
import pickle

In [8]:
class Model:
    def __init__(self, model_params: ModelParams, model_config: ModelConfig):
        self.inner_nodes = model_params.input_nodes;
        self.hidden_layers = model_params.hidden_layers;
        self.outer_nodes = model_params.output_nodes;
        self.optimizer = model_params.optimizer;
        self.loss = model_params.loss;

        self.data_source = model_config.data_source
        self.model_store = model_config.model_store

        self.model = models.Sequential([
            layers.Input(shape=(self.inner_nodes,)),
            layers.Dense(32, activation='relu'),
            layers.Dense(64, activation='relu'),
            layers.Dense(32, activation='relu'),
            layers.Dense(1, activation='sigmoid')
        ]);

    def compile_model_and_store(self):
        self.model.compile(
            optimizer = self.optimizer,
            loss = self.loss,
            metrics=["accuracy", "precision", "recall"]
        );

        os.makedirs(self.model_store, exist_ok=True);
        model_path = os.path.join(str(self.model_store), "model.pkl");

        with open(model_path, 'wb') as file:
            pickle.dump(self.model, file);

    

In [9]:
from exceptions.ModelBuildException import ModelBuildException

In [11]:
import yaml
import sys

In [18]:
try:
    with open('config.yaml') as file:
        data = yaml.safe_load(file);
    with open('params.yaml') as file:
        params = yaml.safe_load(file);

    model_para = params["model_params"];
    model_confi = data["model"];

    model_params: ModelParams = ModelParams(model_para["input_nodes"], model_para['hidden_layers'], model_para['output_nodes'], model_para['optimizer'], model_para['loss']);
    model_config: ModelConfig = ModelConfig(model_confi['data_source'], model_confi['model_store']);

    model = Model(model_params, model_config);
    model.compile_model_and_store();
except Exception as e:
    raise ModelBuildException(e, sys);